# Triggered Backtest Runner

This notebook is triggered by `Coordinator.trigger_backtest()` via the backtest queue file on the VM.

**PM Instructions:**
1. A `/backtest <strategy>` Telegram command (or `coordinator.trigger_backtest()`) writes a job to `/tmp/backtest-queue.json` on the VM.
2. Paste and run these cells in Colab to process the queue.
3. Results are saved to Google Drive and logged back to the Coordinator via `push_alert`.


In [ ]:
# Cell 1: Configuration — edit these values before running
SSH_KEY_FILE = 'ict-bot-ovm-private.key'   # must be uploaded to Colab session
VM_USER      = 'ubuntu'
VM_HOST      = '141.145.193.91'   # Ampere live trader (ict-bot-arm) since the 2026-06-14 cutover
REPO_DIR     = '/home/ubuntu/ict-trading-bot'
QUEUE_FILE   = f'{REPO_DIR}/tmp/backtest-queue.json'

print(f'VM: {VM_USER}@{VM_HOST}:{REPO_DIR}')

In [ ]:
# Cell 2: Pull the backtest queue from the VM
import subprocess, json

result = subprocess.run(
    ['ssh', '-i', SSH_KEY_FILE,
     '-o', 'StrictHostKeyChecking=no',
     f'{VM_USER}@{VM_HOST}',
     f'tail -n 10 {QUEUE_FILE} 2>/dev/null || echo "[]"'],
    capture_output=True, text=True
)

queue = [json.loads(line) for line in result.stdout.strip().splitlines() if line.strip()]
print(f'Queue entries: {len(queue)}')
for i, job in enumerate(queue):
    print(f'  [{i}] {job.get("strategy")} {job.get("symbol")} {job.get("timeframe")} @ {job.get("queued_at")}')

In [ ]:
# Cell 3: Clone/update the repo and run the backtest for the latest job
if not queue:
    print('Queue is empty — nothing to run.')
else:
    job = queue[-1]   # process most recent
    strategy  = job.get('strategy', 'ict')
    symbol    = job.get('symbol', 'BTCUSDT')
    timeframe = job.get('timeframe', '1h')
    start     = job.get('start_date', '2026-01-01')

    print(f'Running backtest: {strategy} {symbol} {timeframe} from {start}')

    cmd = (
        f'cd {REPO_DIR} && '
        f'PYTHONPATH=. python scripts/run_backtest.py '
        f'--strategy {strategy} --symbol {symbol} '
        f'--timeframe {timeframe} --start {start}'
    )
    out = subprocess.run(
        ['ssh', '-i', SSH_KEY_FILE, '-o', 'StrictHostKeyChecking=no',
         f'{VM_USER}@{VM_HOST}', cmd],
        capture_output=True, text=True
    )
    print(out.stdout[-3000:] if out.stdout else '(no output)')
    if out.returncode != 0:
        print('STDERR:', out.stderr[-1000:])

In [ ]:
# Cell 4: Clear processed jobs from the queue on the VM
# Run after confirming backtest succeeded
clear_cmd = f'> {QUEUE_FILE}'   # truncates the file
subprocess.run(
    ['ssh', '-i', SSH_KEY_FILE, '-o', 'StrictHostKeyChecking=no',
     f'{VM_USER}@{VM_HOST}', clear_cmd]
)
print('Queue cleared.')